# Tutorial 5: error mitigation (Qiskit)

*Near-term Quantum Computing 2026-2027, tutorial 5.*

In Lecture 5, we met two methods of error mitigation: zero-noise extrapolation (ZNE) and symmetry verification.
In this tutorial, we run circuits on a simulated copy of a real device, IQM's 20-qubit Garnet, and we measure and mitigate its readout errors.
Then we check two pieces of mitigation code that contain mistakes: ZNE of the energy of Tutorial 4, and symmetry verification of the light cone of Lecture 3.

In this tutorial you will learn to:

1. compile and run circuits on a model of a device, and choose the qubits they run on;
2. measure readout errors, and mitigate them in every shot;
3. read the code of a mitigation method, and find where it does not do what the method needs;
4. judge a mitigated result: how much bias it removes, and how much it increases the standard error.

**How this tutorial works.** This tutorial has one block.

- **Part A** (about 20 minutes): we code together. Type the code yourself in the empty cells.
- **Part B**: three exercises, alone or in pairs. Exercise 1 is a short coding exercise on readout errors.
  In Exercises 2 and 3, you find the mistakes in given code, as in Exercise 3 of Tutorial 4.

**Conventions**, as in Tutorial 4: the TFIM on an open chain of $n = 4$ qubits, $H = -J\sum_i Z_iZ_{i+1} - h\sum_i X_i$, with $J = 1$ and $h = 0.5$.
An energy estimate with "$N$ shots" uses $N$ shots in each of the two measurement settings.
The qubits of a circuit are numbered from 0, as in Qiskit.

Documentation: [Qiskit API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit), [`transpile`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/compiler#transpile), and IQM's [Qiskit user guide](https://docs.meetiqm.com/iqm-client/user_guide_qiskit.html), which describes the device models ("fake backends").

## Setup

In Google Colab, run the next cell first to install the packages.
If you work on your own computer, install the same packages in your Python environment instead: install `iqm-client` without its `[qiskit]` extra, which would replace Qiskit by an older version.

In [ ]:
%pip install -q "qiskit~=2.5.2" "qiskit-aer~=0.18.0" "iqm-client==36.0.0" pylatexenc

import sys
import IPython.core.display
# Colab has an older IPython than iqm-client expects; this line lets iqm-client be imported there
sys.modules.setdefault("IPython.core.display_functions", IPython.core.display)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector, SparsePauliOp

J, h = 1.0, 0.5

The next cell is given: functions of Tutorials 3 and 4 that we reuse.
`tfim(n_qubits, J, h)` builds the Hamiltonian, `signs(bitstring)` turns the bits of one shot into signs $\pm 1$, `group_values` gives the value of each shot in a measurement setting, and `hva_ansatz(n_qubits, n_layers)` is the HVA of Tutorial 4.

In [ ]:
def tfim(n_qubits, J, h):
    """The TFIM on an open chain of n_qubits qubits, as a SparsePauliOp."""
    terms = []
    for i in range(n_qubits - 1):                      # bonds
        terms.append(("ZZ", [i, i + 1], -J))
    for i in range(n_qubits):                          # fields
        terms.append(("X", [i], -h))
    return SparsePauliOp.from_sparse_list(terms, num_qubits=n_qubits)


def signs(bitstring):
    """The outcomes of one shot as signs: signs(bitstring)[q] is +1 or -1 for qubit q."""
    return [1 - 2 * int(b) for b in bitstring[::-1]]   # reversed: Qiskit writes qubit 0 last


def group_values(counts, setting, J, h):
    """One value per shot: s_Z if setting is "Z", s_X if setting is "X"."""
    values = []
    for bitstring, count in counts.items():
        spins = signs(bitstring)
        n_qubits = len(spins)
        if setting == "Z":
            total = 0
            for i in range(n_qubits - 1):
                total += spins[i] * spins[i + 1]
            value = -J * total
        else:
            value = -h * sum(spins)
        values += [value] * count
    return np.array(values)


def hva_ansatz(n_qubits, n_layers):
    """The HVA with n_layers layers on n_qubits qubits; its parameters are (γ_1, β_1, ..., γ_p, β_p)."""
    theta = ParameterVector("theta", 2 * n_layers)
    circuit = QuantumCircuit(n_qubits)
    circuit.h(range(n_qubits))                          # |+...+>
    for k in range(n_layers):
        for i in range(0, n_qubits - 1, 2):             # bonds (0,1), (2,3), ...
            circuit.rzz(theta[2 * k], i, i + 1)
        for i in range(1, n_qubits - 1, 2):             # bonds (1,2), (3,4), ...
            circuit.rzz(theta[2 * k], i, i + 1)
        for i in range(n_qubits):                       # fields
            circuit.rx(theta[2 * k + 1], i)
    return circuit

Our circuit, `state_circuit`, is the HVA with 3 layers, at the angles that COBYLA found with exact energies in Tutorial 4, A2.
Without noise, its energy is 0.003 above $E_0$.
With the depolarising noise of Tutorial 4, Exercise 4 (0.1% per `r` gate and 0.5% per CZ), it was 0.28 above $E_0$.

In [ ]:
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
angles = [-0.655, -1.331, -1.291, -1.137, -1.264, -0.574]   # found in Tutorial 4, A2
state_circuit = hva_ansatz(n_qubits, 3).assign_parameters(angles)
exact_energy = Statevector(state_circuit).expectation_value(hamiltonian).real
print("E0 =", round(E0, 4), "  exact E - E0 =", round(exact_energy - E0, 4))
print(state_circuit.draw(fold=-1))

## Part A: coding together

### A1. A model of a device

Garnet is IQM's superconducting device with 20 qubits on a square grid.
The package `iqm-client` contains a simulated copy of it, `IQMFakeGarnet`.
It is a *backend*: an object that compiles and runs circuits as the device does, here on your computer.
It knows the device's qubits, its native gates, which pairs of qubits can run a CZ (the *coupling map*), and its gate and readout errors.
These error rates were measured once on the device and never change in the model; on the real device, they change from day to day.

![Garnet's coupling map, with the error probability of each CZ, and the best, worst and default layouts of the chain of Tutorial 4 (figure from the notes of Lecture 5)](https://stefanopolla.com/ntqc/figures/qubit-selection.svg)

In this figure from the notes of Lecture 5, each circle is a qubit, QB1 to QB20, and each diamond is a pair of qubits that can run a CZ, with the CZ's error probability.
The three coloured paths are three choices of qubits for the 4-qubit chain of Tutorial 4.
Below the circuit is, for each choice, its circuit fault rate $\lambda$: the expected number of errors in one run.

**Qubit numbers.** Qiskit numbers the qubits from 0, while IQM names them from QB1, as in the figure: Qiskit's qubit 12 is QB13.
`garnet.qubit_name_to_index("QB13")` is 12, and `garnet.index_to_qubit_name(12)` is `"QB13"`.

`IQMFakeGarnet()` creates the backend.
Its number of qubits, native operations and coupling map are `garnet.num_qubits`, `garnet.operation_names` and `garnet.coupling_map`.

In [ ]:
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet

# Create the backend `garnet`, and print its number of qubits, native operations and coupling map
# YOUR CODE HERE

The native gates are `r` (the PRX of Tutorial 2) and `cz`; `delay` and `id` make a qubit wait.
Each pair in the coupling map is a diamond in the figure: for example, the chain QB13, QB14, QB18, QB19 is the pairs [12, 13], [17, 13] and [17, 18].

### A2. Compiling and running

`transpile(circuit, backend=garnet, optimization_level=1)` compiles a circuit for Garnet.
It rewrites the circuit in Garnet's native gates, and chooses which qubit of the device runs each qubit of the circuit: this choice is the *layout*.
If the circuit needs a CZ between two qubits that are not coupled, the compiler adds SWAP gates, which exchange the states of two coupled qubits.

`native.count_ops()` counts the gates of a compiled circuit `native`, and `native.layout.final_index_layout()` lists the device qubits it runs on.
`garnet.run(native, shots=1000).result().get_counts()` runs it with Garnet's errors, as `simulator.run` did in Tutorial 1.
The bits in the counts are in the order of the qubits of *your* circuit (qubit 0 last), whichever device qubits ran them.

In [ ]:
circuit = state_circuit.copy()
circuit.measure_all()
# 1. Compile `circuit` for garnet; call the result `native`.
# 2. Print its gate counts and the device qubits it runs on.
# 3. Run `native` with 1000 shots, and print the counts.
# YOUR CODE HERE

The compiler follows the coupling map, but it does not know the error rates, so it does not know which qubits are best (notes, section ["Choosing the qubits"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-qubit-selection)).
To choose the qubits ourselves, we give the layout to `transpile`: with `initial_layout=[12, 13, 17, 18]`, qubit 0 of the circuit runs on device qubit 12 (QB13), qubit 1 on 13 (QB14), and so on.
On the chain QB13, QB14, QB18, QB19, the CZs of our circuit have the smallest errors (figure above).
This holds for the fixed error rates of the model only: for a run on the real device, choose the qubits from its current calibration data, which IQM Resonance shows.

Our circuit has 9 $R_{ZZ}$ gates, of 2 CZs each: 18 CZs in all.
We compare this with the number of CZs after compiling on two layouts.

In [ ]:
# 1. Make the list `chain` of the Qiskit numbers of QB13, QB14, QB18, QB19 (use garnet.qubit_name_to_index),
#    and the list `first` of the Qiskit numbers of QB1, QB2, QB3, QB4.
# 2. Compile `circuit` with initial_layout=chain, and again with initial_layout=first.
#    Print the gate counts of both.
# YOUR CODE HERE

On the chain, every $R_{ZZ}$ acts on coupled qubits, so the compiled circuit has 18 CZs.
On QB1 to QB4, QB2 and QB3 are not coupled, so the compiler adds two SWAP gates, of 3 CZs each: 24 CZs.
More CZs mean more errors: after compiling for a device, always check the number of CZs.

### A3. The energy on Garnet

The function `garnet_energy` below is `measure_energy` of Tutorial 4, for Garnet: for each of the two settings, it compiles the circuit on a given layout and runs it on `garnet`.
Two lines are missing.

In [ ]:
def garnet_energy(state_circuit, shots, layout):
    """Estimate <H> of the TFIM on garnet, with qubit i of the circuit on the physical qubit layout[i].

    Uses `shots` shots in each of the two settings; returns the estimate and its standard error."""
    n_qubits = state_circuit.num_qubits
    energy, variance = 0, 0
    for setting in ["Z", "X"]:
        circuit = state_circuit.copy()
        if setting == "X":
            circuit.h(range(n_qubits))
        circuit.measure_all()
        # Compile `circuit` for garnet with initial_layout=layout; call the result `native`.
        # Run `native` with `shots` shots; call its counts `counts`.
        # YOUR CODE HERE
        values = group_values(counts, setting, J, h)
        energy += values.mean()
        variance += values.var() / shots
    return energy, np.sqrt(variance)

Now estimate the energy of `state_circuit` on Garnet.

In [ ]:
# Estimate the energy of state_circuit with garnet_energy, with 10_000 shots on the layout `chain`.
# Print the estimate minus E0, and its standard error.
# YOUR CODE HERE

**Think.** With the depolarising noise of Tutorial 4, this circuit was 0.28 above $E_0$.
Why is it further from $E_0$ on Garnet?

*Your answer:*

### Summary: running on a device model

| Task | Qiskit |
| --- | --- |
| the model of Garnet | `garnet = IQMFakeGarnet()`, from `iqm.qiskit_iqm.fake_backends.fake_garnet` |
| its qubits, native operations and coupled pairs | `garnet.num_qubits`, `garnet.operation_names`, `garnet.coupling_map` |
| compile for it | `native = transpile(circuit, backend=garnet, optimization_level=1)` |
| ... on chosen qubits | `transpile(..., initial_layout=[12, 13, 17, 18])` |
| check the compiled circuit | `native.count_ops()`, `native.layout.final_index_layout()` |
| run it, with Garnet's noise | `garnet.run(native, shots=1000).result().get_counts()` (not `AerSimulator.from_backend(garnet)`: it has no noise) |

## Part B: exercises

Exercise 1 is a short coding exercise: we measure Garnet's readout errors and mitigate them.

In Exercises 2 and 3, a mitigation method is coded for you, followed by a report that draws conclusions from its output.
The code runs, but it contains **two mistakes** in how it applies the method, and they make the report wrong.
The mistakes are about the method, not about Python.
For each claim of the report, ask where its number comes from, and compare it with what the notes say the method should give.
Change and re-run parts of the code to test your guesses.

Run Exercise 1 before Exercise 2, which uses its readout errors `e0`, `e1` and its function `garnet_energy_mitigated`.
Exercise 3 needs only `garnet`, from A1.
Shots are random, so your numbers will differ a little from those in the text.

### Exercise 1: readout errors

Readout errors happen once, at the end of the circuit, so they are the easiest errors to mitigate (notes, section ["Mitigating readout errors"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-readout-mitigation)).
Say qubit $q$ reads 1 instead of 0 with probability $e_0$, and 0 instead of 1 with probability $e_1$.
Then the noiseless $\langle Z\rangle$ follows from the measured one, $\langle Z\rangle_{\text{read}}$, as
$$
\langle Z\rangle = \frac{\langle Z\rangle_{\text{read}} - (e_1 - e_0)}{1 - e_0 - e_1} .
$$
This formula is linear, so we can apply it to the sign $s = \pm 1$ of each qubit in each shot: $s \to (s - (e_1 - e_0))/(1 - e_0 - e_1)$.
If the readout errors of different qubits are independent, the products of these mitigated signs average to the noiseless $\langle Z_iZ_{i+1}\rangle$, and the spread of the shot values gives the right standard error.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
angles = [-0.655, -1.331, -1.291, -1.137, -1.264, -0.574]   # found in Tutorial 4, A2
state_circuit = hva_ansatz(n_qubits, 3).assign_parameters(angles)
chain = [12, 13, 17, 18]                                     # QB13, QB14, QB18, QB19
shots = 10_000

**(a) (code)** Measure the readout errors of the qubits of `chain`, as in Tutorial 2, Exercise 2.
The circuit `zeros` only measures $|0000\rangle$, and `ones` flips all qubits and measures $|1111\rangle$.
Compile each on `chain`, and run it with `shots` shots.
Then `e0[q]` is the fraction of the shots of `zeros` in which qubit `q` reads 1, and `e1[q]` the fraction of the shots of `ones` in which it reads 0.
The last lines print the readout errors stored in the model: your values should agree with them to about 0.002, the size of the shot noise.

In [ ]:
zeros = QuantumCircuit(n_qubits)
zeros.measure_all()
ones = QuantumCircuit(n_qubits)
ones.x(range(n_qubits))
ones.measure_all()
e0 = np.zeros(n_qubits)            # e0[q]: the fraction of the shots of `zeros` in which qubit q reads 1
e1 = np.zeros(n_qubits)            # e1[q]: the fraction of the shots of `ones` in which qubit q reads 0
# Compile and run `zeros` and `ones` on `chain`, and fill in e0 and e1 from their counts
# YOUR CODE HERE
print("measured e0:   ", np.round(e0, 4))
print("measured e1:   ", np.round(e1, 4))
calibration = garnet.error_profile.readout_errors   # by qubit name: "0" is p(1|0), "1" is p(0|1)
print("calibration e0:", [calibration[garnet.index_to_qubit_name(q)]["0"] for q in chain])
print("calibration e1:", [calibration[garnet.index_to_qubit_name(q)]["1"] for q in chain])

**(b) (code)** Complete `mitigated_values`: it is `group_values`, with the signs of each shot replaced by their mitigated values.
`np.array(signs(bitstring))` is an array, and `-` and `/` act on arrays entry by entry, so one line does it for all qubits at once.
The function `garnet_energy_mitigated` is given: it is `garnet_energy` with `mitigated_values` in place of `group_values`.
The last lines estimate the energy without and with readout mitigation.

In [ ]:
def mitigated_values(counts, setting, J, h, e0, e1):
    """As group_values, with the sign of each qubit q mitigated for its readout errors e0[q], e1[q]."""
    values = []
    for bitstring, count in counts.items():
        # spins: the signs of this shot, with the readout errors mitigated (one line)
        # YOUR CODE HERE
        if setting == "Z":
            total = 0
            for i in range(len(spins) - 1):
                total += spins[i] * spins[i + 1]
            value = -J * total
        else:
            value = -h * sum(spins)
        values += [value] * count
    return np.array(values)


def garnet_energy_mitigated(state_circuit, shots, layout, e0, e1):
    """As garnet_energy, with the readout errors e0, e1 of the qubits of `layout` mitigated in every shot."""
    n_qubits = state_circuit.num_qubits
    energy, variance = 0, 0
    for setting in ["Z", "X"]:
        circuit = state_circuit.copy()
        if setting == "X":
            circuit.h(range(n_qubits))
        circuit.measure_all()
        native = transpile(circuit, backend=garnet, optimization_level=1, initial_layout=layout)
        counts = garnet.run(native, shots=shots).result().get_counts()
        values = mitigated_values(counts, setting, J, h, e0, e1)
        energy += values.mean()
        variance += values.var() / shots
    return energy, np.sqrt(variance)


energy, error = garnet_energy(state_circuit, shots, chain)
print("without readout mitigation: E - E0 =", round(energy - E0, 3), "±", round(error, 3))
energy, error = garnet_energy_mitigated(state_circuit, shots, chain, e0, e1)
print("with readout mitigation:    E - E0 =", round(energy - E0, 3), "±", round(error, 3))

**(c)** How much of the bias of A3 does readout mitigation remove, and how much is left?
Why is the standard error a little larger with mitigation?

*Your answer:*

### Exercise 2: zero-noise extrapolation

The second cell below applies ZNE to the energy of `state_circuit` on Garnet, with the steps of the notes (section ["Implementing ZNE"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-zne-steps)):

1. compile the circuit into the native gates `r` and `cz`;
2. fold the compiled circuit $U$ into $U(U^\dagger U)^k$, for the noise scale factors $c = 2k + 1 = 1, 3, 5$ (notes, section ["Scaling the noise by folding"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-folding));
3. for each $c$, compile the folded circuit again on the chain of Part A, and estimate its energy with `garnet_energy`, with 10 000 shots;
4. extrapolate to $c = 0$: linearly through $c = 1, 3$, and with Richardson's method through $c = 1, 3, 5$, each with its standard error (notes, sections ["Extrapolating to zero noise"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-extrapolation) and ["The price: amplified shot noise"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-zne-variance)).

One mistake is in the function `fold`, and one in the loop over $c$.

**Your task.**
(a) Find each mistake, and explain in one or two sentences why it is wrong.
(b) Copy the second cell into the empty cell after the report, fix it, run it, and rewrite the report.
Compare your result with the figure ["Zero-noise extrapolation of the energy"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#fig-zne) of the notes, which used the depolarising noise of Tutorial 4.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
angles = [-0.655, -1.331, -1.291, -1.137, -1.264, -0.574]   # found in Tutorial 4, A2
state_circuit = hva_ansatz(n_qubits, 3).assign_parameters(angles)
chain = [12, 13, 17, 18]                                     # QB13, QB14, QB18, QB19

In [ ]:
# ZNE of the energy on the model of Garnet. This cell contains two conceptual mistakes.
shots = 10_000
scale_factors = [1, 3, 5]


def fold(circuit, scale_factor):
    """The circuit U folded into U (U^dagger U)^k, with scale_factor = 2k + 1."""
    folded = circuit.copy()
    for _ in range((scale_factor - 1) // 2):
        folded.compose(circuit.inverse(), inplace=True)
        folded.compose(circuit, inplace=True)
    return folded


native = transpile(state_circuit, basis_gates=["r", "cz"], optimization_level=1)   # step 1
energies, standard_errors = [], []
for c in scale_factors:
    # Folding scales all the noise of the circuit, readout errors included, so the extrapolation
    # to c = 0 removes the readout errors too: garnet_energy is enough, without readout mitigation.
    energy, error = garnet_energy(fold(native, c), shots, chain)
    energies.append(energy)
    standard_errors.append(error)

E1, E3, E5 = energies
s1, s3, s5 = standard_errors
linear = (3 * E1 - E3) / 2                                   # through c = 1, 3
linear_error = np.sqrt((3 / 2 * s1)**2 + (1 / 2 * s3)**2)
richardson = (15 * E1 - 10 * E3 + 3 * E5) / 8               # through c = 1, 3, 5
richardson_error = np.sqrt((15 / 8 * s1)**2 + (10 / 8 * s3)**2 + (3 / 8 * s5)**2)

for c, energy, error in zip(scale_factors, energies, standard_errors):
    print("c =", c, "                   E - E0 =", round(energy - E0, 3), "±", round(error, 3))
print("linear (c = 1, 3):        E - E0 =", round(linear - E0, 3), "±", round(linear_error, 3))
print("Richardson (c = 1, 3, 5): E - E0 =", round(richardson - E0, 3), "±", round(richardson_error, 3))

plt.errorbar(scale_factors, np.array(energies) - E0, yerr=standard_errors, fmt="o", label="measured")
plt.errorbar([0], [linear - E0], yerr=[linear_error], fmt="s", label="linear")
plt.errorbar([0.15], [richardson - E0], yerr=[richardson_error], fmt="^", label="Richardson")
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("noise scale factor c")
plt.ylabel("E - E0")
plt.legend()
plt.show()

**Report.**

1. *Scaling the noise.* The energy does not rise with the scale factor: at $c = 1$, 3 and 5, the
   energies differ only by shot noise. Even when the circuit runs five times, Garnet's gate errors
   hardly change its energy.
2. *Extrapolation.* The two models agree with each other and with the energy at $c = 1$, so the
   extrapolation is reliable (step 5 of ["Implementing ZNE"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-zne-steps)).
   The bias of about 0.6 therefore does not come from the gates, and ZNE cannot reduce it on this
   device.

*Your answer:*

In [ ]:
# Your corrected version of the cell above
# YOUR CODE HERE

### Exercise 3: symmetry verification

In Lecture 3, a flipped qubit spread through a Heisenberg chain as a light cone.
In Lecture 5, noise blurred the cone, and keeping only the shots that pass a symmetry check (post-selection) brought it back (notes, figure ["The light cone with noise"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#fig-light-cone-noise)).
Here we repeat this on Garnet with 7 qubits, the middle one flipped at the start, and up to 5 Trotter steps of length 0.2: fewer than in the notes, so that enough shots pass the check.

The first two cells are correct: they define the circuits (with `heisenberg_step` of Tutorial 3, Exercise 4) and a function that plots the light cones.
The third cell runs the circuits and post-selects the shots, with the steps of the notes (section ["Implementing symmetry verification"](https://stefanopolla.github.io/ntqc/L5-mitigation.html#sec-sv-steps)).
Its function `flip_probabilities` estimates, for each qubit, the probability of reading 1 and its standard error; with `post_select=True`, it uses only the shots that pass the check.
Both mistakes are in `flip_probabilities`: one in which shots it keeps, and one in the standard errors it returns.

**Your task.**
(a) Find each mistake, and explain in one or two sentences why it is wrong.
(b) Copy the third cell into the empty cell after the report, fix it, run it, and rewrite the report.
Compare your result with the figure of the notes.

In [ ]:
# The problem, defined again so that this exercise stands alone
def heisenberg_step(n_qubits, J, dt):
    """One Trotter step of length dt for the Heisenberg chain on n_qubits qubits (Tutorial 3, Exercise 4)."""
    step = QuantumCircuit(n_qubits)
    for first in [0, 1]:                      # bonds (0,1), (2,3), ..., then (1,2), (3,4), ...
        for i in range(first, n_qubits - 1, 2):
            step.rxx(2 * J * dt, i, i + 1)
            step.ryy(2 * J * dt, i, i + 1)
            step.rzz(2 * J * dt, i, i + 1)
    return step


def light_cone_circuit(n_qubits, J, dt, steps):
    """The middle qubit flipped, then `steps` Trotter steps of length dt."""
    circuit = QuantumCircuit(n_qubits)
    circuit.x(n_qubits // 2)
    for _ in range(steps):
        circuit.compose(heisenberg_step(n_qubits, J, dt), inplace=True)
    return circuit


n_qubits, dt, n_steps = 7, 0.2, 5
chain = [4, 9, 8, 7, 12, 13, 17]          # QB5, QB10, QB9, QB8, QB13, QB14, QB18: a chain of coupled qubits
shots = 4000

In [ ]:
from matplotlib.colors import LogNorm      # a logarithmic colour scale, as in Tutorial 3


def plot_light_cones(ideal, noisy, post_selected, kept):
    """The probability of a flip on each qubit against time (without noise, on Garnet, and on Garnet
    with post-selection), and the fraction of shots kept."""
    times = dt * np.arange(n_steps + 1)
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
    titles = ["no noise", "Garnet", "Garnet, post-selected"]
    for ax, data, title in zip(axes[:3], [ideal, noisy, post_selected], titles):
        mesh = ax.pcolormesh(times, range(n_qubits), np.maximum(data, 1e-3).T, shading="nearest",
                             norm=LogNorm(vmin=1e-3, vmax=1))
        ax.set_ylim(n_qubits - 0.5, -0.5)   # qubit 0 at the top, as in a circuit drawing
        ax.set_xlabel("time t")
        ax.set_title(title)
    axes[0].set_ylabel("qubit i")
    fig.colorbar(mesh, ax=axes[:3], label="probability of a flip")
    axes[3].plot(times, kept, "o-")
    axes[3].set_ylim(0, 1)
    axes[3].set_xlabel("time t")
    axes[3].set_title("fraction of shots kept")
    plt.show()

In [ ]:
# Symmetry verification of the light cone on Garnet. This cell contains two conceptual mistakes.

def flip_probabilities(counts, post_select):
    """The probability of reading 1 on each qubit, its standard error, and the fraction of shots used.

    With post_select=True, only the shots that pass the symmetry check are used."""
    ones = np.zeros(n_qubits)          # for each qubit, the number of used shots that read 1
    used, total = 0, 0
    for bitstring, count in counts.items():
        bits = [int(b) for b in bitstring[::-1]]      # bits[q] is the bit of qubit q (Qiskit writes qubit 0 last)
        total += count
        keep = True
        if post_select:
            # The check: every gate conserves the number of flipped qubits, and so its parity. At the
            # start it is odd (one flipped qubit), so we keep the shot if it is still odd.
            keep = sum(bits) % 2 == 1
        if keep:
            ones += count * np.array(bits)
            used += count
    probabilities = ones / used
    standard_errors = np.sqrt(probabilities * (1 - probabilities) / total)
    return probabilities, standard_errors, used / total


ideal, noisy, post_selected = [], [], []                # one row per time: a probability per qubit
noisy_errors, post_selected_errors, kept = [], [], []
for k in range(n_steps + 1):
    circuit = light_cone_circuit(n_qubits, J, dt, k)
    state = Statevector(circuit)                       # without noise, for comparison
    ideal.append([state.probabilities([i])[1] for i in range(n_qubits)])
    circuit.measure_all()
    native = transpile(circuit, backend=garnet, optimization_level=2, initial_layout=chain)   # 3 CZs per bond gate
    counts = garnet.run(native, shots=shots).result().get_counts()
    probabilities, errors, fraction = flip_probabilities(counts, post_select=False)
    noisy.append(probabilities)
    noisy_errors.append(errors)
    probabilities, errors, fraction = flip_probabilities(counts, post_select=True)
    post_selected.append(probabilities)
    post_selected_errors.append(errors)
    kept.append(fraction)
ideal, noisy, post_selected = np.array(ideal), np.array(noisy), np.array(post_selected)
noisy_errors, post_selected_errors = np.array(noisy_errors), np.array(post_selected_errors)

print("time t:                              ", dt * np.arange(n_steps + 1))
print("fraction of shots kept:              ", np.round(kept, 2))
print("flipped qubits per shot, noisy:      ", np.round(noisy.sum(axis=1), 2))
print("flipped qubits per kept shot:        ", np.round(post_selected.sum(axis=1), 2))
print("mean |error|, noisy:                 ", np.round(np.abs(noisy - ideal).mean(axis=1), 3))
print("mean |error|, post-selected:         ", np.round(np.abs(post_selected - ideal).mean(axis=1), 3))
middle = n_qubits // 2
print("middle qubit at t =", dt * n_steps, ": noisy", round(noisy[-1, middle], 3), "±", round(noisy_errors[-1, middle], 4),
      "  post-selected", round(post_selected[-1, middle], 3), "±", round(post_selected_errors[-1, middle], 4),
      "  no noise", round(ideal[-1, middle], 3))
plot_light_cones(ideal, noisy, post_selected, kept)

**Report.**

1. *The check.* Every gate conserves the number of flipped qubits, and post-selection keeps the
   shots in which its parity is that of the initial state. It therefore discards every shot in
   which an error changed the number of flipped qubits. What it cannot catch are errors that leave
   this number unchanged, such as $Z$ errors, and these explain the error that remains after
   post-selection.
2. *The price.* Post-selection discards a growing fraction of the shots, but its estimates are as
   precise as the noisy ones: on the middle qubit at $t = 1$, for example, both standard errors
   are about 0.007. Post-selection therefore reduces the bias at no cost in shots.

*Your answer:*

In [ ]:
# Your corrected version of the cell above
# YOUR CODE HERE

## Wrap-up

- A backend such as `IQMFakeGarnet()` compiles and runs circuits as the device does.
  Choose the qubits with `initial_layout`, and count the CZs of the compiled circuit to check that no SWAPs were added.
- Readout errors are measured with two calibration circuits and mitigated in every shot.
  Folding does not scale them: it repeats the gates, not the measurements.
- Fold the compiled circuit with barriers between its parts, or the compiler removes $U^\dagger U$ and the noise does not grow.
  ZNE then removes most of the bias, and amplifies the shot noise.
- Post-select on the strongest check that the shots allow, and compute the standard errors from the number of kept shots.
- Check a mitigated result against what the method must give: the energy must rise with the scale factor, and every kept shot must have the quantum number of the initial state.

Next week, in Tutorial 6, we run circuits on IQM's devices themselves.